# Parte 2 — Modelo Descriptivo y Diagnóstico
## Proyecto Grupal: AndinaRetail S.A.C.
**Asignatura:** Analítica de Datos | **Rol:** Analista Estadístico/Descriptivo

**Objetivo general:** explicar el comportamiento histórico de AndinaRetail (qué pasó) e
identificar causas (por qué pasó), mediante análisis retrospectivo, segmentación de clientes
y diagnóstico de la caída de margen observada en algunas plazas.

**Preguntas de negocio que aborda esta parte:**
- ¿Qué patrones históricos y estacionales explican el desempeño?
- ¿Qué segmentos de clientes existen?
- ¿Por qué cae el margen en ciertas plazas?


## 1. Configuración inicial y carga de datos

**Objetivo:** preparar el entorno y cargar los datasets sintéticos necesarios para el análisis
descriptivo y diagnóstico.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


In [ ]:
ruta = '../datos/'

tiendas = pd.read_csv(ruta + 'tiendas.csv')
productos = pd.read_csv(ruta + 'productos.csv')
clientes = pd.read_csv(ruta + 'clientes.csv')
ventas = pd.read_csv(ruta + 'ventas.csv', parse_dates=['fecha'])
inventario = pd.read_csv(ruta + 'inventario.csv')

clientes['fecha_registro'] = pd.to_datetime(clientes['fecha_registro'])
if 'fecha_ultima_compra' in clientes.columns:
    clientes['fecha_ultima_compra'] = pd.to_datetime(clientes['fecha_ultima_compra'])

print("Tiendas:", tiendas.shape)
print("Productos:", productos.shape)
print("Clientes:", clientes.shape)
print("Ventas:", ventas.shape)
print("Inventario:", inventario.shape)
print("\nColumnas de ventas:", ventas.columns.tolist())


**Nota:** el dataset `ventas.csv` (versión regenerada por el equipo) ya incluye las columnas
`costo_unitario`, `margen_unitario` y `margen_total` calculadas, por lo que no es necesario
derivarlas manualmente desde `productos`. Se validó previamente (Notebook 01 / exploración) que
estas columnas son 100% consistentes con la fórmula `margen_total = monto_total - costo_unitario × cantidad`
(diferencia máxima ≈ 6×10⁻¹², atribuible a redondeo de punto flotante).

## 2. Preparación de datos: unión de tablas y variables de tiempo

**Objetivo:** construir una tabla analítica (`ventas_full`) que combine ventas con categoría de
producto y ciudad/tipo de tienda, y derivar variables temporales para el análisis de series de
tiempo.

In [ ]:
ventas_full = ventas.merge(
    productos[['id_producto', 'categoria', 'subcategoria']], on='id_producto', how='left'
)
ventas_full = ventas_full.merge(
    tiendas[['id_tienda', 'ciudad', 'tipo']], on='id_tienda', how='left'
)

# Variables de tiempo
ventas_full['anio'] = ventas_full['fecha'].dt.year
ventas_full['mes'] = ventas_full['fecha'].dt.month
ventas_full['anio_mes'] = ventas_full['fecha'].dt.to_period('M')

# Canal agrupado (Físico vs Digital)
ventas_full['canal_tipo'] = np.where(ventas_full['canal'] == 'Tienda', 'Físico', 'Digital')

print(ventas_full.shape)
ventas_full.head(3)


In [ ]:
# Validación de consistencia de margen (heredado de Notebook 01)
ventas_full['margen_calculado'] = ventas_full['monto_total'] - (ventas_full['costo_unitario'] * ventas_full['cantidad'])
diferencia_max = (ventas_full['margen_total'] - ventas_full['margen_calculado']).abs().max()
print("Diferencia máxima entre margen_total y margen calculado:", diferencia_max)

print("\nVentas con margen_total negativo:", (ventas_full['margen_total'] < 0).sum())


**Resultado / Interpretación:** se confirma la consistencia de los datos (diferencia
prácticamente nula). Se identificaron **209 líneas de venta con margen negativo**, un hallazgo
que se investiga en profundidad en la Sección 5 (Diagnóstico de caída de margen).

## 3. Series de tiempo: tendencia y estacionalidad

**Objetivo:** identificar tendencias y patrones estacionales en las ventas, y analizar el
crecimiento del canal digital a lo largo del tiempo.

**Método:** agregación mensual de ventas, comparación de meses del calendario (estacionalidad)
y evolución de la participación del canal digital vs. físico por año.

In [ ]:
serie_ventas = ventas_full.groupby('anio_mes')['monto_total'].sum().reset_index()
serie_ventas['anio_mes_str'] = serie_ventas['anio_mes'].astype(str)

plt.figure(figsize=(16, 6))
plt.plot(serie_ventas['anio_mes_str'], serie_ventas['monto_total'], marker='o')
plt.xticks(rotation=90)
plt.title('Ventas mensuales totales - AndinaRetail (2023-2025)')
plt.ylabel('Monto Total (S/)')
plt.axhline(y=serie_ventas['monto_total'].mean(), color='red', linestyle='--', alpha=0.5, label='Promedio')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Estacionalidad: ventas totales por mes del calendario (2023-2025 combinado)
estacionalidad = ventas_full.groupby('mes')['monto_total'].sum()

plt.figure(figsize=(12, 5))
sns.barplot(x=estacionalidad.index, y=estacionalidad.values, palette='crest')
plt.title('Ventas totales por mes (2023-2025 combinado) — Estacionalidad')
plt.xlabel('Mes')
plt.ylabel('Monto total (S/)')
plt.show()

print(estacionalidad.sort_values(ascending=False))


In [ ]:
# Crecimiento del canal digital en el tiempo
serie_canal = ventas_full.groupby(['anio_mes', 'canal_tipo'])['monto_total'].sum().reset_index()
serie_canal['anio_mes_str'] = serie_canal['anio_mes'].astype(str)

plt.figure(figsize=(16, 6))
for canal in serie_canal['canal_tipo'].unique():
    data = serie_canal[serie_canal['canal_tipo'] == canal]
    plt.plot(data['anio_mes_str'], data['monto_total'], marker='o', label=canal)
plt.xticks(rotation=90)
plt.title('Evolución de ventas: canal Físico vs. Digital')
plt.ylabel('Monto Total (S/)')
plt.legend()
plt.tight_layout()
plt.show()

part_digital = ventas_full.groupby(['anio', 'canal_tipo'])['monto_total'].sum().unstack()
part_digital['%_digital'] = (part_digital['Digital'] / (part_digital['Digital'] + part_digital['Físico']) * 100).round(2)
print(part_digital)


**Resultado / Interpretación:**
- **Estacionalidad confirmada:** los meses de **julio** y **diciembre** presentan los picos más
  altos de ventas, consistentes con Fiestas Patrias y Navidad — el patrón de diseño esperado.
- **Crecimiento sostenido del canal digital:** la participación digital pasó de **35.6% (2023)**
  a **45.8% (2024)** y **55.1% (2025)**, con incrementos anuales de magnitud similar (~10 puntos
  porcentuales). En 2025, por primera vez, el canal digital **superó** al canal físico en monto
  de ventas. Esto valida una tendencia clara y consistente de digitalización del negocio, con
  implicancia directa para la estrategia comercial y de inversión en canales.

## 4. Análisis de Pareto (80/20)

**Objetivo:** identificar los principales contribuyentes al negocio en tres dimensiones:
productos, clientes y categorías.

In [ ]:
# Pareto de productos
ventas_por_producto = ventas_full.groupby('id_producto')['monto_total'].sum().reset_index()
ventas_por_producto = ventas_por_producto.merge(
    productos[['id_producto', 'nombre', 'categoria']], on='id_producto'
).sort_values('monto_total', ascending=False).reset_index(drop=True)

ventas_por_producto['pct_acumulado'] = (ventas_por_producto['monto_total'].cumsum() /
                                          ventas_por_producto['monto_total'].sum() * 100)

n_productos_80 = (ventas_por_producto['pct_acumulado'] <= 80).sum() + 1
pct_productos_80 = n_productos_80 / len(ventas_por_producto) * 100
print(f"{n_productos_80} productos ({pct_productos_80:.1f}% del catálogo) generan el 80% de las ventas")
display(ventas_por_producto.head(10)[['id_producto', 'nombre', 'categoria', 'monto_total', 'pct_acumulado']])


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.bar(range(len(ventas_por_producto)), ventas_por_producto['monto_total'], color='steelblue')
ax1.set_xlabel('Productos (ordenados de mayor a menor venta)')
ax1.set_ylabel('Monto Total (S/)', color='steelblue')

ax2 = ax1.twinx()
ax2.plot(range(len(ventas_por_producto)), ventas_por_producto['pct_acumulado'], color='darkred', linewidth=2)
ax2.axhline(y=80, color='green', linestyle='--', label='80%')
ax2.axvline(x=n_productos_80, color='orange', linestyle='--', label=f'{n_productos_80} productos')
ax2.set_ylabel('% Acumulado', color='darkred')
ax2.legend(loc='center right')

plt.title('Análisis de Pareto - Productos')
plt.tight_layout()
plt.show()


In [ ]:
# Pareto de clientes
ventas_por_cliente = ventas_full.groupby('id_cliente')['monto_total'].sum().sort_values(ascending=False).reset_index()
ventas_por_cliente['pct_acumulado'] = ventas_por_cliente['monto_total'].cumsum() / ventas_por_cliente['monto_total'].sum() * 100

n_clientes_80 = (ventas_por_cliente['pct_acumulado'] <= 80).sum() + 1
pct_clientes_80 = n_clientes_80 / len(ventas_por_cliente) * 100
print(f"{n_clientes_80} clientes ({pct_clientes_80:.1f}% de la base) generan el 80% de las ventas")


In [ ]:
# Pareto de categorías
ventas_por_categoria = ventas_full.groupby('categoria')['monto_total'].sum().sort_values(ascending=False).reset_index()
ventas_por_categoria['pct_acumulado'] = ventas_por_categoria['monto_total'].cumsum() / ventas_por_categoria['monto_total'].sum() * 100
ventas_por_categoria['pct_del_total'] = ventas_por_categoria['monto_total'] / ventas_por_categoria['monto_total'].sum() * 100

print(ventas_por_categoria.round(2))


**Resultado / Interpretación:**
- **Productos:** el 17.8% del catálogo (142 de 800 productos) genera el 80% de las ventas,
  cercano a la regla clásica 80/20. Los productos de mayor venta pertenecen casi exclusivamente
  a la categoría **Electrohogar**.
- **Clientes:** se requiere el **45.2%** de la base de clientes para alcanzar el 80% de las
  ventas — una concentración mucho más baja de lo esperado en retail. Esto indica que el negocio
  **no depende de un pequeño grupo de clientes VIP**, sino de una base amplia y relativamente
  homogénea, lo cual tiene implicancia directa para el diseño de estrategias de fidelización
  (deben ser masivas, no solo dirigidas a pocos clientes grandes).
- **Categorías:** solo 2 de 6 categorías (**Electrohogar 65.6%** y **Hogar 21.6%**) explican el
  87% del monto de ventas, mientras que Abarrotes y Bebidas —que concentran más transacciones—
  aportan apenas 3% y 2% del valor. El negocio depende críticamente de la categoría Electrohogar,
  lo que representa un riesgo de concentración a vigilar.

## 5. Segmentación de clientes: RFM y clustering (K-Means)

**Objetivo:** segmentar a los clientes según Recencia, Frecuencia y Valor Monetario (RFM), y
validar la segmentación con un enfoque no supervisado (K-Means).

**Método:** cálculo de RFM con fecha de referencia 2025-12-31 (según especificación del
enunciado para el cálculo de churn), asignación de scores por quintiles, definición de
segmentos de negocio, y clustering K-Means sobre las mismas variables estandarizadas.

In [ ]:
fecha_referencia = ventas_full['fecha'].max()
print("Fecha de referencia para RFM:", fecha_referencia)

rfm = ventas_full.groupby('id_cliente').agg(
    recencia=('fecha', lambda x: (fecha_referencia - x.max()).days),
    frecuencia=('id_venta', 'count'),
    valor_monetario=('monto_total', 'sum')
).reset_index()

print(rfm.describe())


In [ ]:
# Scores RFM (quintiles 1-5); recencia se invierte (menor recencia = mejor)
rfm['R_score'] = pd.qcut(rfm['recencia'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frecuencia'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['valor_monetario'], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['RFM_total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

print(rfm[['recencia', 'R_score', 'frecuencia', 'F_score', 'valor_monetario', 'M_score', 'RFM_total']].head(10))


In [ ]:
def segmentar_cliente(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Campeones'
    elif r >= 4 and f >= 3:
        return 'Clientes leales'
    elif r >= 4 and f <= 2:
        return 'Nuevos / potenciales'
    elif r == 3:
        return 'Necesitan atención'
    elif r <= 2 and f >= 4:
        return 'En riesgo (antes valiosos)'
    elif r <= 2 and f <= 2 and m >= 4:
        return 'No se pueden perder'
    else:
        return 'Hibernando / perdidos'

rfm['segmento'] = rfm.apply(segmentar_cliente, axis=1)

print(rfm['segmento'].value_counts())
print("\n% del total:")
print((rfm['segmento'].value_counts(normalize=True) * 100).round(1))


In [ ]:
caracterizacion = rfm.groupby('segmento').agg(
    n_clientes=('id_cliente', 'count'),
    recencia_prom=('recencia', 'mean'),
    frecuencia_prom=('frecuencia', 'mean'),
    valor_prom=('valor_monetario', 'mean')
).round(1).sort_values('valor_prom', ascending=False)

print(caracterizacion)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

orden = caracterizacion.sort_values('n_clientes', ascending=False).index
sns.barplot(x=rfm['segmento'].value_counts().reindex(orden).values, y=orden, ax=axes[0], palette='viridis')
axes[0].set_title('Cantidad de clientes por segmento RFM')
axes[0].set_xlabel('N° de clientes')

sns.barplot(x=caracterizacion['valor_prom'], y=caracterizacion.index, ax=axes[1], palette='rocket')
axes[1].set_title('Valor monetario promedio por segmento')
axes[1].set_xlabel('Valor monetario promedio (S/)')

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(data=rfm, x='recencia', y='frecuencia', hue='segmento',
                 size='valor_monetario', sizes=(10, 200), alpha=0.6, palette='tab10')
plt.title('Segmentación RFM: Recencia vs. Frecuencia')
plt.xlabel('Recencia (días desde última compra)')
plt.ylabel('Frecuencia (n° de compras)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


**Resultado / Interpretación (RFM manual):** la base de clientes se distribuye en 7
segmentos. Los **Campeones** (24.8%, valor promedio S/17,645) representan el motor del negocio,
mientras que el segmento **Hibernando/perdidos** es el más numeroso (35.3% de la base), con
recencia promedio de 253 días y el menor valor histórico. Los segmentos **"No se pueden perder"**
y **"En riesgo"** (4.8% combinado) son de atención prioritaria: representan clientes de alto
valor histórico que muestran señales tempranas de inactivación. Esta segmentación es la base
para el modelo predictivo de churn a desarrollar en la Parte 3, dado que el 35.3% de clientes
en estado "Hibernando/perdidos" es consistente con el patrón de fuga de clientes señalado como
preocupación estratégica por la Gerencia.

### Validación con clustering no supervisado (K-Means)

In [ ]:
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recencia', 'frecuencia', 'valor_monetario']])

inercia = []
rango_k = range(2, 9)
for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inercia.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(rango_k, inercia, marker='o')
plt.xlabel('Número de clusters (k)')
plt.ylabel('Inercia')
plt.title('Método del codo')
plt.show()


In [ ]:
k_optimo = 4
kmeans = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
rfm['cluster_kmeans'] = kmeans.fit_predict(rfm_scaled)

caracterizacion_kmeans = rfm.groupby('cluster_kmeans').agg(
    n_clientes=('id_cliente', 'count'),
    recencia_prom=('recencia', 'mean'),
    frecuencia_prom=('frecuencia', 'mean'),
    valor_prom=('valor_monetario', 'mean')
).round(1).sort_values('valor_prom', ascending=False)

print(caracterizacion_kmeans)
print("\nComparación K-Means vs. segmentos RFM manuales:")
print(pd.crosstab(rfm['cluster_kmeans'], rfm['segmento']))


**Resultado / Interpretación (K-Means):** el método del codo sugiere **k=4** como número
óptimo de clusters (caída pronunciada de inercia hasta k=4, luego se suaviza). Los 4 clusters
resultantes muestran una progresión clara y monótona en las 3 dimensiones RFM:

| Cluster | N° clientes | Recencia (días) | Frecuencia | Valor promedio |
|---|---|---|---|---|
| Alto valor, muy activos | 2,491 | 13.7 | 34.7 | S/21,817 |
| Valor medio, activos | 4,895 | 36.4 | 20.8 | S/9,933 |
| Baja actividad, valor bajo | 6,227 | 131.4 | 9.1 | S/3,543 |
| Prácticamente perdidos | 1,282 | 504.1 | 3.9 | S/1,917 |

La consistencia entre el enfoque supervisado (RFM manual) y el no supervisado (K-Means) refuerza
la validez de la segmentación: ambos métodos identifican de forma independiente una estructura
similar de clientes de alto valor/actividad frente a un núcleo de clientes inactivos.

## 6. Análisis diagnóstico: caída de margen en Trujillo

**Objetivo:** realizar un diagnóstico de causa raíz sobre la caída de margen observada en
algunas plazas, mediante drill-down por ciudad/periodo, comparación de cohortes (antes/después)
y descomposición de la variación (descuento y costo de almacenamiento).

In [ ]:
# Identificación inicial: ¿dónde y cuándo se concentran los márgenes negativos?
margen_negativo = ventas_full[ventas_full['margen_total'] < 0]

print("Márgenes negativos por ciudad:")
print(margen_negativo['ciudad'].value_counts())

print("\nMárgenes negativos por año-mes:")
print(margen_negativo['fecha'].dt.to_period('M').value_counts().sort_index())


**Resultado:** de las 209 ventas con margen negativo, **169 (81%) corresponden a Trujillo**,
concentrándose fuertemente a partir de **abril de 2025 (2025-Q2)** en adelante — antes de esa
fecha, los casos eran esporádicos (máximo 4 por mes).

In [ ]:
# Drill-down: comparación de cohortes Trujillo vs. otras ciudades, antes/después de 2025-Q2
ventas_full['periodo_diagnostico'] = np.where(
    ventas_full['fecha'] >= '2025-04-01', 'Desde 2025-Q2', 'Antes de 2025-Q2'
)
ventas_full['es_trujillo'] = np.where(ventas_full['ciudad'] == 'Trujillo', 'Trujillo', 'Otras ciudades')

resumen_margen = ventas_full.groupby(['es_trujillo', 'periodo_diagnostico']).agg(
    monto_total_sum=('monto_total', 'sum'),
    margen_total_sum=('margen_total', 'sum'),
    descuento_prom=('descuento_pct', 'mean'),
    n_ventas=('id_venta', 'count')
)
resumen_margen['margen_pct'] = (resumen_margen['margen_total_sum'] / resumen_margen['monto_total_sum'] * 100).round(2)
print(resumen_margen)


In [ ]:
# Serie mensual de margen % y descuento %: Trujillo vs. otras ciudades
serie_mensual = ventas_full.groupby(['anio_mes', 'es_trujillo']).apply(
    lambda x: pd.Series({
        'margen_pct': x['margen_total'].sum() / x['monto_total'].sum() * 100,
        'descuento_prom': x['descuento_pct'].mean() * 100
    })
).reset_index()
serie_mensual['anio_mes'] = serie_mensual['anio_mes'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for grupo in serie_mensual['es_trujillo'].unique():
    data = serie_mensual[serie_mensual['es_trujillo'] == grupo]
    axes[0].plot(data['anio_mes'], data['margen_pct'], marker='o', label=grupo)
    axes[1].plot(data['anio_mes'], data['descuento_prom'], marker='o', label=grupo)

axes[0].axvline(x='2025-04', color='red', linestyle='--', alpha=0.5, label='Inicio caída (2025-Q2)')
axes[0].set_title('Margen % mensual: Trujillo vs. Otras ciudades')
axes[0].set_ylabel('Margen %')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=90)

axes[1].axvline(x='2025-04', color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Descuento % promedio mensual: Trujillo vs. Otras ciudades')
axes[1].set_ylabel('Descuento %')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
# Segunda causa: costo de almacenamiento (desde inventario.csv)
inventario['periodo_dt'] = pd.to_datetime(inventario['periodo'] + '-01')
inv_full = inventario.merge(tiendas[['id_tienda', 'ciudad']], on='id_tienda', how='left')
inv_full['es_trujillo'] = np.where(inv_full['ciudad'] == 'Trujillo', 'Trujillo', 'Otras ciudades')
inv_full['periodo_diagnostico'] = np.where(
    inv_full['periodo_dt'] >= '2025-04-01', 'Desde 2025-Q2', 'Antes de 2025-Q2'
)

resumen_almacen = inv_full.groupby(['es_trujillo', 'periodo_diagnostico']).agg(
    costo_unit_prom=('costo_almacenamiento_unitario', 'mean'),
    costo_total_sum=('costo_almacenamiento_total', 'sum')
).round(2)

print(resumen_almacen)


**Resultado / Interpretación — Diagnóstico de la caída de margen en Trujillo:**

Se identificó que, a partir de abril de 2025 (2025-Q2), las tiendas de Trujillo presentan una
caída de margen de **24.33% a 13.86%** (-10.5 puntos porcentuales), mientras que el resto de
ciudades mantuvo un margen estable (~23-24%) en el mismo período. Esta caída se explica por dos
factores concurrentes:

1. **Aumento del descuento promedio:** en Trujillo se triplicó, de 5.69% a 16.85%, muy por
   encima del resto de ciudades (que se mantuvo en 6.5%-7%).
2. **Aumento del costo de almacenamiento:** el costo unitario en Trujillo casi se duplicó, de
   S/0.30 a S/0.56, mientras el resto de ciudades no mostró variación (S/0.30 estable).

El patrón coincide con **209 líneas de venta con margen negativo** en todo el dataset, de las
cuales el **81% (169 casos)** corresponden a Trujillo, concentrándose especialmente en los meses
de octubre a diciembre de 2025.

**Recomendación:** investigar la causa operativa del cambio de política de descuentos y de
costos logísticos en la plaza de Trujillo, dado que ambos factores explican de forma consistente
y cuantificable la pérdida de rentabilidad detectada.

## 7. Conclusiones de negocio

1. **Estacionalidad y canal digital:** las ventas presentan picos claros en julio y diciembre.
   El canal digital muestra un crecimiento sostenido (35.6% → 45.8% → 55.1% de participación
   entre 2023 y 2025), superando ya al canal físico en 2025 — se recomienda reforzar la
   inversión en el canal digital dada su trayectoria de crecimiento.

2. **Concentración del negocio:** el valor de AndinaRetail depende críticamente de la categoría
   **Electrohogar** (65.6% de las ventas) y de un núcleo de 142 productos (17.8% del catálogo).
   En contraste, la base de **clientes está poco concentrada** (se requiere 45.2% de los
   clientes para el 80% de ventas), lo que sugiere que las estrategias de fidelización deben ser
   masivas y no dirigidas solo a un pequeño grupo de clientes VIP.

3. **Segmentación de clientes:** el 35.3% de la base está en estado "Hibernando/perdidos"
   (validado también por clustering K-Means), representando el principal riesgo de fuga de
   clientes de la compañía. Los segmentos "No se pueden perder" y "En riesgo" (4.8% de la base)
   requieren acciones de retención prioritarias por su alto valor histórico. Esta segmentación
   sienta las bases para el modelo predictivo de churn de la Parte 3.

4. **Diagnóstico de margen:** la caída de rentabilidad en Trujillo desde 2025-Q2 (-10.5 puntos
   porcentuales de margen) se explica cuantitativamente por un incremento simultáneo del
   descuento promedio (casi triplicado) y del costo de almacenamiento unitario (casi duplicado),
   aislado a esa plaza y ese período — se recomienda una revisión operativa focalizada en
   Trujillo antes de replicar sus políticas comerciales en otras ciudades.

**Nota de reproducibilidad:** todos los resultados dependen de los archivos CSV generados con
semilla fija. Si el equipo regenera los datos, este notebook debe re-ejecutarse de inicio a fin
(Restart + Run All) para actualizar los resultados numéricos; la lógica y estructura del
análisis permanecen válidas.